In [0]:
print("Manufacturing Performance Analytics")
print("Purpose: Analyse production, quality defects, and machine downtime.")
print("Tools: Databricks, PySpark, SQL, and Power BI")

Manufacturing Performance Analytics
Purpose: Analyse production, quality defects, and machine downtime.
Tools: Databricks, PySpark, SQL, and Power BI


In [0]:
from datetime import date, timedelta
import random

random.seed(42)

machines = ["M001", "M002", "M003", "M004"]
products = ["Passenger Tyre", "Truck Tyre", "Motorcycle Tyre"]

records = []

start_date = date(2026, 1, 1)

for day in range(90):
    production_date = start_date + timedelta(days=day)

    for machine in machines:
        units = random.randint(800, 1500)
        defective = random.randint(10, 70)
        downtime = random.randint(10, 180)

        # Simulate a machine with higher defects and downtime
        if machine == "M003":
            defective = random.randint(60, 120)
            downtime = random.randint(100, 240)

        records.append((
            production_date,
            machine,
            random.choice(products),
            units,
            defective,
            downtime
        ))

columns = [
    "production_date",
    "machine_id",
    "product_type",
    "units_produced",
    "defective_units",
    "downtime_minutes"
]

df = spark.createDataFrame(records, columns)

print("Total records:", df.count())
display(df.limit(10))

Total records: 360


production_date,machine_id,product_type,units_produced,defective_units,downtime_minutes
2026-01-01,M001,Motorcycle Tyre,1454,17,16
2026-01-01,M002,Passenger Tyre,1081,25,67
2026-01-01,M003,Passenger Tyre,904,65,208
2026-01-01,M004,Passenger Tyre,830,15,65
2026-01-02,M001,Motorcycle Tyre,1317,48,16
2026-01-02,M002,Motorcycle Tyre,1003,55,176
2026-01-02,M003,Passenger Tyre,1358,88,171
2026-01-02,M004,Truck Tyre,963,54,118
2026-01-03,M001,Truck Tyre,1084,19,65
2026-01-03,M002,Passenger Tyre,904,15,107


In [0]:
from pyspark.sql import functions as F

print("Total records:", df.count())

# Check missing values
print("Missing values by column:")
df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()

# Check duplicate records
print("Duplicate records:", df.count() - df.dropDuplicates().count())

# Check invalid production values
invalid_records = df.filter(
    (F.col("units_produced") <= 0) |
    (F.col("defective_units") < 0) |
    (F.col("defective_units") > F.col("units_produced")) |
    (F.col("downtime_minutes") < 0)
)

print("Invalid records:", invalid_records.count())

Total records: 360
Missing values by column:
+---------------+----------+------------+--------------+---------------+----------------+
|production_date|machine_id|product_type|units_produced|defective_units|downtime_minutes|
+---------------+----------+------------+--------------+---------------+----------------+
|              0|         0|           0|             0|              0|               0|
+---------------+----------+------------+--------------+---------------+----------------+

Duplicate records: 0
Invalid records: 0


In [0]:
from pyspark.sql import functions as F

machine_kpis = (
    df.groupBy("machine_id")
      .agg(
          F.sum("units_produced").alias("total_units_produced"),
          F.sum("defective_units").alias("total_defects"),
          F.sum("downtime_minutes").alias("total_downtime_minutes")
      )
      .withColumn(
          "defect_rate_pct",
          F.round(
              F.col("total_defects") /
              F.col("total_units_produced") * 100,
              2
          )
      )
      .orderBy(F.desc("defect_rate_pct"))
)

display(machine_kpis)

machine_id,total_units_produced,total_defects,total_downtime_minutes,defect_rate_pct
M003,102375,8063,15057,7.88
M004,101259,3645,8841,3.6
M001,103206,3695,8887,3.58
M002,102494,3489,9148,3.4


In [0]:
from pyspark.sql import functions as F

daily_kpis = (
    df.groupBy("production_date")
      .agg(
          F.sum("units_produced").alias("total_production"),
          F.sum("defective_units").alias("total_defects"),
          F.sum("downtime_minutes").alias("total_downtime_minutes")
      )
      .withColumn(
          "defect_rate_pct",
          F.round(
              F.col("total_defects") / F.col("total_production") * 100,
              2
          )
      )
      .orderBy("production_date")
)

display(daily_kpis)

production_date,total_production,total_defects,total_downtime_minutes,defect_rate_pct
2026-01-01,4269,122,356,2.86
2026-01-02,4641,245,481,5.28
2026-01-03,4425,176,380,3.98
2026-01-04,4190,237,414,5.66
2026-01-05,4826,171,494,3.54
2026-01-06,4328,263,505,6.08
2026-01-07,4343,151,530,3.48
2026-01-08,4363,207,423,4.74
2026-01-09,4828,257,362,5.32
2026-01-10,4283,211,395,4.93


In [0]:

spark.sql("SHOW CATALOGS").show(truncate=False)


+---------+
|catalog  |
+---------+
|samples  |
|system   |
|workspace|
+---------+



In [0]:

spark.sql("SHOW SCHEMAS IN workspace").show(truncate=False)


+------------------+
|databaseName      |
+------------------+
|default           |
|information_schema|
+------------------+



In [0]:

df.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.manufacturing_production"
)

print("Production dataset saved successfully!")


Production dataset saved successfully!


In [0]:

saved_df = spark.table(
    "workspace.default.manufacturing_production"
)

print("Total records:", saved_df.count())
display(saved_df.limit(10))


Total records: 360


production_date,machine_id,product_type,units_produced,defective_units,downtime_minutes
2026-01-01,M001,Motorcycle Tyre,1454,17,16
2026-01-01,M002,Passenger Tyre,1081,25,67
2026-01-01,M003,Passenger Tyre,904,65,208
2026-01-01,M004,Passenger Tyre,830,15,65
2026-01-02,M001,Motorcycle Tyre,1317,48,16
2026-01-02,M002,Motorcycle Tyre,1003,55,176
2026-01-02,M003,Passenger Tyre,1358,88,171
2026-01-02,M004,Truck Tyre,963,54,118
2026-01-03,M001,Truck Tyre,1084,19,65
2026-01-03,M002,Passenger Tyre,904,15,107


In [0]:
%sql
SELECT
    machine_id,
    SUM(units_produced) AS total_production,
    SUM(defective_units) AS total_defects,
    SUM(downtime_minutes) AS total_downtime,
    ROUND(
        SUM(defective_units) * 100.0 /
        SUM(units_produced), 2
    ) AS defect_rate_pct
FROM workspace.default.manufacturing_production
GROUP BY machine_id
ORDER BY defect_rate_pct DESC;


machine_id,total_production,total_defects,total_downtime,defect_rate_pct
M003,102375,8063,15057,7.88
M004,101259,3645,8841,3.60
M001,103206,3695,8887,3.58
M002,102494,3489,9148,3.40


In [0]:

%sql
SELECT
    production_date,
    SUM(units_produced) AS total_production,
    SUM(defective_units) AS total_defects,
    SUM(downtime_minutes) AS total_downtime,
    ROUND(
        SUM(defective_units) * 100.0 /
        SUM(units_produced), 2
    ) AS defect_rate_pct
FROM workspace.default.manufacturing_production
GROUP BY production_date
ORDER BY production_date;


production_date,total_production,total_defects,total_downtime,defect_rate_pct
2026-01-01,4269,122,356,2.86
2026-01-02,4641,245,481,5.28
2026-01-03,4425,176,380,3.98
2026-01-04,4190,237,414,5.66
2026-01-05,4826,171,494,3.54
2026-01-06,4328,263,505,6.08
2026-01-07,4343,151,530,3.48
2026-01-08,4363,207,423,4.74
2026-01-09,4828,257,362,5.32
2026-01-10,4283,211,395,4.93


In [0]:

%sql
SELECT
    COUNT(DISTINCT production_date) AS production_days,
    COUNT(DISTINCT machine_id) AS total_machines,
    SUM(units_produced) AS total_production,
    SUM(defective_units) AS total_defects,
    ROUND(
        SUM(defective_units) * 100.0 /
        SUM(units_produced), 2
    ) AS overall_defect_rate_pct,
    SUM(downtime_minutes) AS total_downtime_minutes,
    ROUND(
        SUM(downtime_minutes) / 60.0, 2
    ) AS total_downtime_hours
FROM workspace.default.manufacturing_production;


production_days,total_machines,total_production,total_defects,overall_defect_rate_pct,total_downtime_minutes,total_downtime_hours
90,4,409334,18892,4.62,41933,698.88


In [0]:

%sql
SELECT
    machine_id,
    SUM(units_produced) AS total_production,
    SUM(defective_units) AS total_defects
FROM workspace.default.manufacturing_production
GROUP BY machine_id
ORDER BY machine_id;


machine_id,total_production,total_defects
M001,103206,3695
M002,102494,3489
M003,102375,8063
M004,101259,3645


Databricks visualization. Run in Databricks to view.

In [0]:

%sql
SELECT
    machine_id,
    SUM(units_produced) AS total_production,
    SUM(defective_units) AS total_defects,
    SUM(downtime_minutes) AS total_downtime_minutes,
    ROUND(
        SUM(defective_units) * 100.0 /
        SUM(units_produced), 2
    ) AS defect_rate_pct,
    ROUND(
        SUM(downtime_minutes) / 60.0, 2
    ) AS downtime_hours
FROM workspace.default.manufacturing_production
GROUP BY machine_id
ORDER BY machine_id;


machine_id,total_production,total_defects,total_downtime_minutes,defect_rate_pct,downtime_hours
M001,103206,3695,8887,3.58,148.12
M002,102494,3489,9148,3.40,152.47
M003,102375,8063,15057,7.88,250.95
M004,101259,3645,8841,3.60,147.35


In [0]:

%sql
SELECT
    production_date,
    SUM(units_produced) AS total_production,
    SUM(defective_units) AS total_defects,
    SUM(downtime_minutes) AS total_downtime_minutes
FROM workspace.default.manufacturing_production
GROUP BY production_date
ORDER BY production_date;


production_date,total_production,total_defects,total_downtime_minutes
2026-01-01,4269,122,356
2026-01-02,4641,245,481
2026-01-03,4425,176,380
2026-01-04,4190,237,414
2026-01-05,4826,171,494
2026-01-06,4328,263,505
2026-01-07,4343,151,530
2026-01-08,4363,207,423
2026-01-09,4828,257,362
2026-01-10,4283,211,395
